# OASBUD preprocessing

In [ ]:
import os
import sys
from pathlib import Path

import pandas as pd
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 80)

cwd = Path.cwd().resolve()
REPO_ROOT = cwd if (cwd / "pyproject.toml").exists() else cwd.parent
assert (REPO_ROOT / "pyproject.toml").exists(), "Could not locate the repository root."
os.chdir(REPO_ROOT)  
sys.path.insert(0, str(REPO_ROOT))

## Load raw data

In [ ]:
from preprocessing.oasbud import Oasbud, USDataHolder, csv_save_path
from utils.preprocessing import birads_assessment, get_value, process_us_from_mat

ds = Oasbud()
ds.set_dataset_name("oasbud")
print(f"raw exam tuples: {len(ds.source_data)}")

### Step 1 — unpack one raw exam tuple

In [ ]:
exam_set = ds.source_data[0]
_, us1, us2, seg1, seg2, birads, _ = exam_set
birads = birads[0]
birads

### Step 2 — demodulate/compress the raw RF-like signal into a displayable US image

In [ ]:
us1_processed = process_us_from_mat(us1)
us2_processed = process_us_from_mat(us2)
us1_processed.shape, us1_processed.dtype

### Step 3 — map the raw BI-RADS code to a harmonized label

In [ ]:
birads_label = get_value(birads, birads_assessment)
birads_label

## Build one exam record per image

In [ ]:
holder1 = USDataHolder(patient_id="0", img=us1_processed, segmentation=seg1, birads=birads_label)
holder2 = USDataHolder(patient_id="0", img=us2_processed, segmentation=seg2, birads=birads_label)
exam1 = ds.process_example(holder1)
exam1

## Final processed CSV

In [ ]:
import json

final_df = pd.read_csv(csv_save_path)
print(f"raw exam tuples: {len(ds.source_data)} (x2 images each) | rows in final csv: {len(final_df)}")
final_df.head()

In [ ]:
sample = final_df.iloc[0]
print(json.dumps(json.loads(sample['context']), indent=2))
print(json.dumps(json.loads(sample['findings']), indent=2))